[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhayraj01op/AI-Post-Studio/blob/main/AI_Post_Studio_Colab.ipynb)

# AI Post Studio — ComfyUI + Google Colab
Images → Posts → Biodata (Apna Connection).

**Kaise chalayein:** Upar menu mein *Runtime → Change runtime type → T4 GPU* chunein, phir *Runtime → Run all*. Aakhri cell ek `gradio.live` link dega — us par click karke app kholein.

Sab kuch (ComfyUI, models, photos, posts, biodata) aapki Google Drive ke `MyDrive/AI_Post_Studio` folder mein save hota hai.

In [ ]:
#@title 1. Settings
GITHUB_REPO = "https://github.com/abhayraj01op/AI-Post-Studio"  #@param {type:"string"}
TEXT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  #@param ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Off (sirf template, sabse fast)"]
PHOTO_MODEL = "RealVisXL V5 (sabse real)"  #@param ["RealVisXL V5 (sabse real)", "Juggernaut XL v9 (achhe kapde/light)"]
# Model badalne par purana model khud delete hota hai, taaki Drive full na ho.
MODELS = {
  "RealVisXL V5 (sabse real)": "https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors",
  "Juggernaut XL v9 (achhe kapde/light)": "https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors",
}
CHECKPOINT_URL = MODELS[PHOTO_MODEL]
BASE = "/content/drive/MyDrive/AI_Post_Studio"


In [ ]:
#@title 2. Google Drive mount
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(BASE, exist_ok=True)
print('Drive ready:', BASE)

In [ ]:
#@title 3. One-click setup (pehli baar ~10-15 min, baad mein ~2 min)
import os, subprocess
def sh(c):
    print('$', c); r = subprocess.run(c, shell=True)
    if r.returncode: raise SystemExit(f'Fail: {c}')
sh('apt-get -qq update && apt-get -qq install -y libraqm0 libfribidi0 aria2 > /dev/null')
# App code (har baar GitHub se latest)
if os.path.exists('/content/aps'): sh('cd /content/aps && git pull -q')
else: sh(f'git clone -q {GITHUB_REPO} /content/aps')
# ComfyUI Drive mein (ek baar install, hamesha save)
comfy = f'{BASE}/ComfyUI'
if not os.path.exists(f'{comfy}/main.py'): sh(f'git clone -q https://github.com/comfyanonymous/ComfyUI "{comfy}"')
else: sh(f'cd "{comfy}" && git pull -q')
sh(f'pip -q install -r "{comfy}/requirements.txt"')
sh('pip -q install -r /content/aps/requirements.txt')
# Photo model (Drive mein ek baar download)
ck = f'{comfy}/models/checkpoints'; os.makedirs(ck, exist_ok=True)
name = CHECKPOINT_URL.split('/')[-1].split('?')[0]
# Sirf ek photo model rakho (dusra ~7 GB delete) -> Drive full nahi hoga
for f in os.listdir(ck):
    if f.endswith('.safetensors') and f != name:
        print('Purana model hata rahe:', f); os.remove(f'{ck}/{f}')
if not os.path.exists(f'{ck}/{name}'): sh(f'aria2c -q -c -x 16 -s 16 --file-allocation=none -d "{ck}" -o "{name}" "{CHECKPOINT_URL}"')
# Quality boosters (chhote files): 4x upscaler (67 MB) + T4-safe VAE (335 MB)
um = f'{comfy}/models/upscale_models'; os.makedirs(um, exist_ok=True)
if not os.path.exists(f'{um}/4x-UltraSharp.pth'): sh(f'wget -q -O "{um}/4x-UltraSharp.pth" https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth')
vd = f'{comfy}/models/vae'; os.makedirs(vd, exist_ok=True)
if not os.path.exists(f'{vd}/sdxl_vae_fp16_fix.safetensors'): sh(f'wget -q -O "{vd}/sdxl_vae_fp16_fix.safetensors" https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors')
# Face fix (FaceDetailer) — chehre bigadne se bachata hai
cn = f'{comfy}/custom_nodes'
for repo in ['ComfyUI-Impact-Pack', 'ComfyUI-Impact-Subpack']:
    if not os.path.exists(f'{cn}/{repo}'): sh(f'git clone -q https://github.com/ltdrdata/{repo} "{cn}/{repo}"')
    else: sh(f'cd "{cn}/{repo}" && git pull -q')
    if os.path.exists(f'{cn}/{repo}/requirements.txt'): sh(f'pip -q install -r "{cn}/{repo}/requirements.txt"')
bb = f'{comfy}/models/ultralytics/bbox'; os.makedirs(bb, exist_ok=True)
if not os.path.exists(f'{bb}/face_yolov8m.pt'): sh(f'wget -q -O "{bb}/face_yolov8m.pt" https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt')
# Hindi fonts
fd = f'{BASE}/fonts'; os.makedirs(fd, exist_ok=True)
for w in ['ExtraBold','Bold','Medium']:
    if not os.path.exists(f'{fd}/Mukta-{w}.ttf'):
        sh(f'wget -q -O "{fd}/Mukta-{w}.ttf" https://github.com/google/fonts/raw/main/ofl/mukta/Mukta-{w}.ttf')
os.environ.update(APS_BASE=BASE, APS_CHECKPOINT=name, APS_TEXT_MODEL=TEXT_MODEL, HF_HOME='/content/hf_cache', APS_HF_CACHE='/content/hf_cache')
# Purana 15GB broken text model Drive se hatao (jagah + time bachega)
import shutil; shutil.rmtree(f'{BASE}/hf_cache', ignore_errors=True)
# Purani ComfyUI copies Drive se hatao (app apni copy alag save karta hai)
shutil.rmtree(f'{BASE}/data/comfy_output', ignore_errors=True)
sh('pip cache purge > /dev/null 2>&1 || true')
os.environ['APS_COMFY_OUT'] = '/content/comfy_out'
print('Setup poora ho gaya ✔')

In [ ]:
#@title 4. Name test (4 din wala rule)
!cd /content/aps && python -m pytest -q tests

In [ ]:
#@title 5. Start — ComfyUI + App (link neeche aayega)
import subprocess, time, urllib.request, os
log = open('/content/comfyui.log', 'w')
subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188',
                  '--output-directory', '/content/comfy_out', '--preview-method', 'none'], cwd=f'{BASE}/ComfyUI', stdout=log, stderr=log)
for _ in range(120):
    try: urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2); print('ComfyUI chal raha ✔'); break
    except Exception: time.sleep(2)
else: print('ComfyUI start nahi hua, /content/comfyui.log dekhein')
!cd /content/aps && python -m app.main

**Tips**
- Colab band hone par bhi sab Drive mein safe hai. Agli baar bas *Run all*.
- ComfyUI ka apna UI bhi chahiye? `ComfyUI` ko `--listen 0.0.0.0` se chalakar cloudflared tunnel use karein (optional).
- Free Colab GPU kuch ghante baad band ho sakta hai; Colab Pro se lamba chalta hai.
- Drive full dikhe to drive.google.com par **Trash (Bin) khali karein** — delete hui files wahan tab tak jagah gherti hain.
- v6: har photo mein do alag chehre mila kar bilkul naya chehra banta hai; upscaler + face fix se photo saaf aati hai.